#### kb chatbot, retrieval

In [19]:
import boto3
import json
import pandas as pd

# AWSセッション
session = boto3.Session(
    profile_name="bedrock-dev",
    region_name="ap-northeast-1"
)

# Bedrock Knowledge Bases Runtime
bedrock_agent_runtime = session.client(
    service_name="bedrock-agent-runtime"
)

# Knowledge Base ID
KB_ID = "PW4SFGINEE"#"Your KB ID"

In [ ]:
import os
import boto3

# AWSプロファイルを指定
os.environ["AWS_PROFILE"] = "bedrock-dev"
os.environ["AWS_DEFAULT_REGION"] = "ap-northeast-1"

# Session作成
session = boto3.Session()

# IAMユーザー確認
sts = session.client("sts")
identity = sts.get_caller_identity()

print("Profile:", session.profile_name)
print("ARN:", identity["Arn"])

In [12]:

query = "2026年10月18日の残課題を教えてください"

response = bedrock_agent_runtime.retrieve(
    knowledgeBaseId=KB_ID,
    retrievalQuery={
        "text": query
    },
    retrievalConfiguration={
        "managedSearchConfiguration": {
            "numberOfResults": 5
        }
    }
)

results = response["retrievalResults"]

print(f"取得件数: {len(results)}")

for i, result in enumerate(results, 1):
    print(f"\n--- 検索結果 {i} ---")
    print(result["content"].get("text", ""))
    print("スコア:", result.get("score"))


取得件数: 5

--- 検索結果 1 ---
# 2026-10-18 日報：外出による気分転換と学習方法の整理

## 目的

外出して気分をリフレッシュし、休日の過ごし方を過ごした。

また、GraphRAGの品質改善方法について調査するとともに、今後の学習効率を高めるため、インプットとアウトプットの整理方法を明確にする。

## 実施内容・作業時間

| 内容 | 作業時間 | 実施内容 |
|---|---:|---|
| 外出 | 4時間30分 | 御誕生寺、越前たけふ駅、お多福食堂、西山公園を訪問し、ゴルフの打ちっぱなしを実施 |
| クラッシュ・ロワイヤル | 3時間30分 | ゲームをプレイ |
| VIVANTの視聴 | 1時間 | ドラマを視聴 |
スコア: 0.6356469988822937

--- 検索結果 2 ---
今後は、インプットだけで終わらせず、アウトプットを通じて理解度を確認することを意識したい。特にプログラミング学習では、実際にコードを動かし、その仕組みや動作結果を説明できる状態を目指す。

### 残課題

**1. GraphRAGの改善方法を実装・検証する**

今回はWeb検索による調査が中心となったため、調査した改善方法を実際のプログラムに適用し、効果を確認する必要がある。まずはエンティティ抽出やリレーション抽出の改善に取り組み、変更前後のナレッジグラフを比較しながら、品質への影響を検証したい。

**2.
スコア: 0.6075007915496826

--- 検索結果 3 ---
- 複数の質問を入力し、それぞれの回答を取得できることを確認する
- 質問に対する回答内容が、参照元の文書と整合しているか確認する
- 回答の根拠となる参照情報や引用情報を取得できるか確認する


昨日：[2026-10-17](2026-10-17)
スコア: 0.5120139122009277

--- 検索結果 4 ---
今後は、評価指標ごとの意味を理解したうえで、実際にRagasによる評価を実行し、検索結果や生成回答の品質を数値で確認できる仕組みを作りたい。

### 残課題

**1. Azure Blob Storageからのデータ取得と、RAG処理の接続**

Blob Storageへの接続とデータ取得について

In [3]:

for i, result in enumerate(results, 1):
    print(f"\n===== 検索結果 {i} =====")

    metadata = result.get("metadata", {})

    if metadata:
        for key, value in metadata.items():
            print(f"{key}: {value}")
    else:
        print("metadataはありません")



===== 検索結果 1 =====
date: 2026-10-18
_source_uri: https://amzn-bedrock-demo01.s3.ap-northeast-1.amazonaws.com/markdown/2026-10-18.md
date_confidence: high
_chunk_id: ljrtAWn9LiQhczii9iHgJWsw-tEu90kXL4sz2hZhExE
_data_source_type: S3
_language_code: ja
_created_at: 2026-11-12T03:30:28Z
_data_source_id: JWCTJMX5QT
_file_type: PLAIN_TEXT
_document_title: 2026-10-18.md
category_confidence: middle
_last_updated_at: 2026-11-12T03:30:28Z
category: 振り返り

===== 検索結果 2 =====
date: 2026-10-18
_source_uri: https://amzn-bedrock-demo01.s3.ap-northeast-1.amazonaws.com/markdown/2026-10-18.md
date_confidence: high
_chunk_id: VogTJ0Bu0uMQKrcdDYq7rxWoeoNWfsHFXRjIEIzHyHs
_data_source_type: S3
_language_code: ja
_created_at: 2026-11-12T03:30:28Z
_data_source_id: JWCTJMX5QT
_file_type: PLAIN_TEXT
_document_title: 2026-10-18.md
category_confidence: middle
_last_updated_at: 2026-11-12T03:30:28Z
category: 振り返り

===== 検索結果 3 =====
date: 2026-10-18
_source_uri: https://amzn-bedrock-demo01.s3.ap-northeast-1.amazon

In [4]:

import pandas as pd

rows = []

for i, result in enumerate(results, 1):
    row = {
        "順位": i,
        "スコア": result.get("score"),
        "本文": result.get("content", {}).get("text", ""),
        "参照元": result.get("location", {})
    }

    # metadataを列として追加
    row.update(result.get("metadata", {}))

    rows.append(row)

df = pd.DataFrame(rows)

display(df)


,順位,スコア,本文,参照元,date,_source_uri,date_confidence,_chunk_id,_data_source_type,_language_code,_created_at,_data_source_id,_file_type,_document_title,category_confidence,_last_updated_at,category
0,1,0.635647,# 2026-10-18 日報：外出による気分転換と学習方法の整理\n\n## 目的\n\n...,{'s3Location': {'uri': 'https://amzn-bedrock-d...,2026-10-18,https://amzn-bedrock-demo01.s3.ap-northeast-1....,high,ljrtAWn9LiQhczii9iHgJWsw-tEu90kXL4sz2hZhExE,S3,ja,2026-11-12T03:30:28Z,JWCTJMX5QT,PLAIN_TEXT,2026-10-18.md,middle,2026-11-12T03:30:28Z,振り返り
1,2,0.607501,今後は、インプットだけで終わらせず、アウトプットを通じて理解度を確認することを意識したい。特...,{'s3Location': {'uri': 'https://amzn-bedrock-d...,2026-10-18,https://amzn-bedrock-demo01.s3.ap-northeast-1....,high,VogTJ0Bu0uMQKrcdDYq7rxWoeoNWfsHFXRjIEIzHyHs,S3,ja,2026-11-12T03:30:28Z,JWCTJMX5QT,PLAIN_TEXT,2026-10-18.md,middle,2026-11-12T03:30:28Z,振り返り
2,3,0.512014,- 複数の質問を入力し、それぞれの回答を取得できることを確認する\n- 質問に対する回答内容...,{'s3Location': {'uri': 'https://amzn-bedrock-d...,2026-10-18,https://amzn-bedrock-demo01.s3.ap-northeast-1....,high,NviempDkYLk0cm7bMVZdJOF1OPDbEQVhjTYBwuuK8Us,S3,ja,2026-11-12T03:30:28Z,JWCTJMX5QT,PLAIN_TEXT,2026-10-18.md,middle,2026-11-12T03:30:28Z,振り返り
3,4,0.484019,今後は、評価指標ごとの意味を理解したうえで、実際にRagasによる評価を実行し、検索結果や生...,{'s3Location': {'uri': 'https://amzn-bedrock-d...,2026-10-16,https://amzn-bedrock-demo01.s3.ap-northeast-1....,high,w8gZNhP0RCHl3VDjT-c1_96SG9LVyUIlUwd6UV_LoNI,S3,ja,2026-11-12T03:30:29Z,JWCTJMX5QT,PLAIN_TEXT,2026-10-16.md,middle,2026-11-12T03:30:29Z,クラウド
4,5,0.461500,Reactは便利な機能を提供する一方、基礎知識が不足していると、その利点やプログラムの構成を...,{'s3Location': {'uri': 'https://amzn-bedrock-d...,2026-10-17,https://amzn-bedrock-demo01.s3.ap-northeast-1....,high,RqNr_2jy6sNIHdFMBT18HyxakPB7l_cdzHoUMe7p72k,S3,ja,2026-11-12T03:30:29Z,JWCTJMX5QT,PLAIN_TEXT,2026-10-17.md,middle,2026-11-12T03:30:29Z,振り返り


#### metadata filter retrieval

In [5]:
query = "残課題を教えてください"

response = bedrock_agent_runtime.retrieve(
    knowledgeBaseId=KB_ID,
    retrievalQuery={
        "text": query
    },
    retrievalConfiguration={
        "managedSearchConfiguration": {
            "numberOfResults": 5,
            "filter": {
                "equals": {
                    "key": "date",
                    "value": "2026-10-18"
                }
            }
        }
    }
)

results = response["retrievalResults"]

print(f"取得件数: {len(results)}")

for i, result in enumerate(results, 1):
    print(f"\n--- 検索結果 {i} ---")
    print(result["content"].get("text", ""))
    print("スコア:", result.get("score"))

取得件数: 5

--- 検索結果 1 ---
今後は、インプットだけで終わらせず、アウトプットを通じて理解度を確認することを意識したい。特にプログラミング学習では、実際にコードを動かし、その仕組みや動作結果を説明できる状態を目指す。

### 残課題

**1. GraphRAGの改善方法を実装・検証する**

今回はWeb検索による調査が中心となったため、調査した改善方法を実際のプログラムに適用し、効果を確認する必要がある。まずはエンティティ抽出やリレーション抽出の改善に取り組み、変更前後のナレッジグラフを比較しながら、品質への影響を検証したい。

**2.
スコア: 0.6310436129570007

--- 検索結果 2 ---
**2. インプットとアウトプットを組み合わせた学習を実践する**

インプットとアウトプットの役割は整理できたが、実際の学習で十分に活用できているかは、今後確認する必要がある。学習した内容について、メモやコードリーディングで仕組みを理解した後、プログラムの実装やMarkdownでの整理を行い、理解と成果の両方を確認したい。


## 次回に向けて

**1. テストプログラムのハンズオン**

Pythonでテストプログラムを作成し、実際に実行しながらテストの基本的な仕組みを理解する。

- 正常系・異常系のテストケースを作成する
- `assert`を使用して、期待する結果と実際の結果を比較する
スコア: 0.2304070144891739

--- 検索結果 3 ---
# 2026-10-18 日報：外出による気分転換と学習方法の整理

## 目的

外出して気分をリフレッシュし、休日の過ごし方を過ごした。

また、GraphRAGの品質改善方法について調査するとともに、今後の学習効率を高めるため、インプットとアウトプットの整理方法を明確にする。

## 実施内容・作業時間

| 内容 | 作業時間 | 実施内容 |
|---|---:|---|
| 外出 | 4時間30分 | 御誕生寺、越前たけふ駅、お多福食堂、西山公園を訪問し、ゴルフの打ちっぱなしを実施 |
| クラッシュ・ロワイヤル | 3時間30分 | ゲームをプレイ |
| VIVANTの視聴 | 1時間 | ドラマを視聴 |
スコア: 0.21336244

#### retrieval(kb) + generation(myself), not metadata filter

In [11]:
# settings
MODEL_ID = "jp.anthropic.claude-haiku-4-5-20251001-v1:0"

bedrock_agent_runtime = boto3.client(
    "bedrock-agent-runtime",
    region_name="ap-northeast-1"
)

bedrock_runtime = boto3.client(
    "bedrock-runtime",
    region_name="ap-northeast-1"
)

# question
query = "2026年10月17日の残課題を教えて"

# retrieval
response = bedrock_agent_runtime.retrieve(
    knowledgeBaseId=KB_ID,
    retrievalQuery={
        "text": query
    },
    retrievalConfiguration={
        "managedSearchConfiguration": {
            "numberOfResults": 5
        }
    }
)

results = response["retrievalResults"]
print(f"取得件数: {len(results)}")

# retrieval results changes context
context = "\n\n".join(
    result["content"].get("text", "")
    for result in results
)

# making prompt
prompt = f"""
あなたは情報を整理するアシスタントです。

以下の参考情報をもとに質問に回答してください。

【参考情報】
{context}

【質問】
{query}

【回答ルール】
・参考情報に記載されている内容を根拠に回答する。
・参考情報にない情報は推測しない。
・情報が不足している場合は、その旨を伝える。
・日本語で回答する。
"""

# LLM answer generation
generation_response = bedrock_runtime.converse(
    modelId=MODEL_ID,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "text": prompt
                }
            ]
        }
    ],
    inferenceConfig={
        "temperature": 0
    }
    
)

answer = generation_response["output"]["message"]["content"][0]["text"]

# result
print("===Answer===")
print(answer)

取得件数: 5
===Answer===
# 2026年10月17日の残課題

参考情報に基づいて、2026年10月17日の残課題は以下の通りです：

## 1. 学習内容の実践と理解の確認

インプットだけで終わらせず、アウトプットを通じて理解度を確認することを意識する必要があります。特にプログラミング学習では、実際にコードを動かし、その仕組みや動作結果を説明できる状態を目指すことが課題です。

## 2. GraphRAGの改善方法を実装・検証する

Web検索による調査が中心となったため、調査した改善方法を実際のプログラムに適用し、効果を確認する必要があります。具体的には：
- エンティティ抽出やリレーション抽出の改善に取り組む
- 変更前後のナレッジグラフを比較しながら、品質への影響を検証する

## 3. 残課題（記載不完全）

参考情報では「**2.」の項目が途中で切れており、完全な内容が記載されていません。

---

**注記：** 参考情報に記載されている残課題は上記の通りですが、「2.」の項目が不完全なため、全体像の把握には情報が不足しています。


#### retrieval(kb) + generation(kb), not metadata filter

In [21]:
import boto3

bedrock_agent_runtime = boto3.client(
    "bedrock-agent-runtime",
    region_name="ap-northeast-1"
)


query = "2026年10月18日の残課題を教えてください"

response = bedrock_agent_runtime.agentic_retrieve_stream(
    messages=[
        {
            "role": "user",
            "content": {
                "text": query
            }
        }
    ],
    retrievers=[
        {
            "configuration": {
                "knowledgeBase": {
                    "knowledgeBaseId": KB_ID
                }
            }
        }
    ],
    agenticRetrieveConfiguration={
        "foundationModelType": "MANAGED",
        "rerankingModelType": "MANAGED"
    },
    generateResponse=True
)

# イベントストリームから回答を取得
for event in response["stream"]:
    if "result" in event:
        result = event["result"]

        if "generatedResponse" in result:
            print("回答:")
            print(result["generatedResponse"]["answer"])

        print("\n参照元:")
        for item in result.get("results", []):
            print(item.get("content"))

回答:
2026年10月18日の日報における「残課題」は以下の2点です。

## 残課題

**1. GraphRAGの改善方法を実装・検証する**
今回はWeb検索による調査が中心だったため、調査した改善方法を実際のプログラムに適用し、効果を確認する必要がある。まずはエンティティ抽出やリレーション抽出の改善に取り組み、変更前後のナレッジグラフを比較しながら、品質への影響を検証したい。

**2. インプットとアウトプットを組み合わせた学習を実践する**
インプットとアウトプットの役割は整理できたが、実際の学習で十分に活用できているかは今後確認する必要がある。学習した内容について、メモやコードリーディングで仕組みを理解した後、プログラムの実装やMarkdownでの整理を行い、理解と成果の両方を確認したい。

なお、次回に向けての予定としては、「テストプログラムのハンズオン」（pytestでの正常系・異常系テストやモックの利用確認など）と「Azure KBのNotebookでの動作確認」（Azure AI Searchを用いたRAGの回答取得・整合性・根拠情報の確認）が挙げられています。

参照元:
{'mimeType': 'text/plain', 'text': '# 2026-10-18 日報：外出による気分転換と学習方法の整理\n\n## 目的\n\n外出して気分をリフレッシュし、休日の過ごし方を過ごした。\n\nまた、GraphRAGの品質改善方法について調査するとともに、今後の学習効率を高めるため、インプットとアウトプットの整理方法を明確にする。\n\n## 実施内容・作業時間\n\n| 内容 | 作業時間 | 実施内容 |\n|---|---:|---|\n| 外出 | 4時間30分 | 御誕生寺、越前たけふ駅、お多福食堂、西山公園を訪問し、ゴルフの打ちっぱなしを実施 |\n| クラッシュ・ロワイヤル | 3時間30分 | ゲームをプレイ |\n| VIVANTの視聴 | 1時間 | ドラマを視聴 |'}
{'mimeType': 'text/plain', 'text': '今後は、インプットだけで終わらせず、アウトプットを通じて理解度を確認することを意識したい。特にプログラミング学習では、実際にコードを動かし、その仕組みや動作結果を説明できる状態を

#### retrieval(kb) + generation(kb), metadata filter

In [22]:
import boto3

bedrock_agent_runtime = boto3.client(
    "bedrock-agent-runtime",
    region_name="ap-northeast-1"
)


query = "2026年10月18日の残課題を教えてください"

response = bedrock_agent_runtime.agentic_retrieve_stream(
    messages=[
        {
            "role": "user",
            "content": {
                "text": query
            }
        }
    ],
    retrievers=[
        {
            "configuration": {
                "knowledgeBase": {
                    "knowledgeBaseId": KB_ID,

                    # Metadata Filter
                    "retrievalOverrides": {
                        "filter": {
                            "equals": {
                                "key": "date",
                                "value": "2026-10-18"
                            }
                        },
                        "maxNumberOfResults": 5
                    }
                }
            }
        }
    ],
    agenticRetrieveConfiguration={
        "foundationModelType": "MANAGED",
        "rerankingModelType": "MANAGED"
    },
    generateResponse=True
)

# イベントストリームから回答を取得
for event in response["stream"]:
    if "result" in event:
        result = event["result"]

        if "generatedResponse" in result:
            print("回答:")
            print(result["generatedResponse"]["answer"])

        print("\n参照元:")
        for item in result.get("results", []):
            print(item.get("content"))

回答:
2026年10月18日の日報に記載されている「残課題」は以下の2点です。

- **GraphRAGの改善方法を実装・検証する**：今回はWeb検索による調査が中心となったため、調査した改善方法を実際のプログラムに適用し、効果を確認する必要がある。まずはエンティティ抽出やリレーション抽出の改善に取り組み、変更前後のナレッジグラフを比較しながら、品質への影響を検証する。
- **インプットとアウトプットを組み合わせた学習を実践する**：インプットとアウトプットの役割は整理できたが、実際の学習で十分に活用できているかは今後確認する必要がある。学習した内容について、メモやコードリーディングで仕組みを理解した後、プログラムの実装やMarkdownでの整理を行い、理解と成果の両方を確認する。

参照元:
{'mimeType': 'text/plain', 'text': '# 2026-10-18 日報：外出による気分転換と学習方法の整理\n\n## 目的\n\n外出して気分をリフレッシュし、休日の過ごし方を過ごした。\n\nまた、GraphRAGの品質改善方法について調査するとともに、今後の学習効率を高めるため、インプットとアウトプットの整理方法を明確にする。\n\n## 実施内容・作業時間\n\n| 内容 | 作業時間 | 実施内容 |\n|---|---:|---|\n| 外出 | 4時間30分 | 御誕生寺、越前たけふ駅、お多福食堂、西山公園を訪問し、ゴルフの打ちっぱなしを実施 |\n| クラッシュ・ロワイヤル | 3時間30分 | ゲームをプレイ |\n| VIVANTの視聴 | 1時間 | ドラマを視聴 |'}
{'mimeType': 'text/plain', 'text': '今後は、インプットだけで終わらせず、アウトプットを通じて理解度を確認することを意識したい。特にプログラミング学習では、実際にコードを動かし、その仕組みや動作結果を説明できる状態を目指す。\n\n### 残課題\n\n**1. GraphRAGの改善方法を実装・検証する**\n\n今回はWeb検索による調査が中心となったため、調査した改善方法を実際のプログラムに適用し、効果を確認する必要がある。まずはエンティティ抽出やリレーション抽出の改善に取り組み、変更前後のナレッジグラフを